# Metrics Executor — Test Notebook NEEDS UPDATING TO ACCOUNT FOR NFLREADPY

This notebook validates `metrics_executor.py` end-to-end on the 2025 regular season PBP data.


1. Import libraries & the executor
2. Load & clean PBP (same logic as your main notebook)
3. Run smoke tests & display outputs



In [1]:
# Auto-reload so edits to metrics_executor.py are picked up
%load_ext autoreload
%autoreload 2

In [2]:
import polars as pl
import nfl_data_py as nfl
from metrics_executor import MetricsExecutor
print('Libraries imported. Ready to load data.')

Libraries imported. Ready to load data.


### **Load and Clean Data**

In [3]:
# --- Clean 2025 Play-by-Play Data using Polars ---

# Load 2025 play-by-play data
df_twofive = pl.from_pandas(
    nfl.import_pbp_data(
        [2025],
        downcast=True,
        cache=False,
        alt_path=None,
        include_participation=False
    )
)

# Keep only regular season plays
df_twofive = df_twofive.filter(pl.col("season_type") == "REG")

# Normalize pass/rush flags
df_twofive = df_twofive.with_columns([
    pl.col("pass").fill_null(0).cast(pl.Int8).alias("pass"),
    pl.col("rush").fill_null(0).cast(pl.Int8).alias("rush"),
])

# Overwrite play_type pass/run if flagged, else keep original
#     Also handle any numeric or null values 
df_twofive = df_twofive.with_columns(
    pl.when(pl.col("pass") == 1).then(pl.lit("pass"))
     .when(pl.col("rush") == 1).then(pl.lit("run"))
     .otherwise(
         pl.col("play_type")
           .cast(pl.Utf8)
           .fill_null("no_play")
           .replace({"0": "no_play", "0.0": "no_play"})
     )
     .alias("play_type")
)

#Clean up whitespace / case for consistency
df_twofive = df_twofive.with_columns(
    pl.col("play_type").str.strip_chars().str.to_lowercase()
)



2025 done.
Downcasting floats.


## Test Data Cleaning

#Checking that plays include epa, null values are handled, etc

In [4]:
# --- Clean 2025 Play-by-Play Data using Polars ---

# 1️⃣ Load 2025 play-by-play data
df_twofive = pl.from_pandas(
    nfl.import_pbp_data(
        [2025],
        downcast=True,
        cache=False,
        alt_path=None,
        include_participation=False
    )
)

# 2️⃣ Keep only regular season plays
df_twofive = df_twofive.filter(pl.col("season_type") == "REG")

# 3️⃣ Normalize pass/rush flags (handle nulls, enforce integer dtype)
df_twofive = df_twofive.with_columns([
    pl.col("pass").fill_null(0).cast(pl.Int8).alias("pass"),
    pl.col("rush").fill_null(0).cast(pl.Int8).alias("rush"),
])

# 4️⃣ Overwrite play_type like pandas: pass/run if flagged, else keep original
#     Also handle any numeric or null values safely.
df_twofive = df_twofive.with_columns(
    pl.when(pl.col("pass") == 1).then(pl.lit("pass"))
     .when(pl.col("rush") == 1).then(pl.lit("run"))
     .otherwise(
         pl.col("play_type")
           .cast(pl.Utf8)
           .fill_null("no_play")
           .replace({"0": "no_play", "0.0": "no_play"})
     )
     .alias("play_type")
)

# 5️⃣ (Optional) Clean up whitespace / case for consistency
df_twofive = df_twofive.with_columns(
    pl.col("play_type").str.strip_chars().str.to_lowercase()
)

# 6️⃣ Final sanity check
print("Rows total:", df_twofive.height)
print("Unique play types:", df_twofive.select(pl.col("play_type").unique()).to_series().to_list())
print("Null EPA:", df_twofive.select(pl.col("epa").null_count()).item())

# 7️⃣ Preview value counts like pandas
df_twofive.select(pl.col("play_type").value_counts()).to_pandas()

2025 done.
Downcasting floats.
Rows total: 16193
Unique play types: ['pass', 'kickoff', 'no_play', 'extra_point', 'punt', 'qb_spike', 'field_goal', 'qb_kneel', 'run']
Null EPA: 188


,play_type
0,"{'play_type': 'qb_kneel', 'count': 158}"
1,"{'play_type': 'punt', 'count': 677}"
2,"{'play_type': 'qb_spike', 'count': 34}"
3,"{'play_type': 'run', 'count': 4510}"
4,"{'play_type': 'field_goal', 'count': 395}"
5,"{'play_type': 'no_play', 'count': 1593}"
6,"{'play_type': 'extra_point', 'count': 438}"
7,"{'play_type': 'pass', 'count': 7420}"
8,"{'play_type': 'kickoff', 'count': 968}"


In [5]:
#check play type totals

df_twofive.select(pl.col("play_type").value_counts()).to_pandas()

,play_type
0,"{'play_type': 'no_play', 'count': 1593}"
1,"{'play_type': 'field_goal', 'count': 395}"
2,"{'play_type': 'punt', 'count': 677}"
3,"{'play_type': 'qb_kneel', 'count': 158}"
4,"{'play_type': 'qb_spike', 'count': 34}"
5,"{'play_type': 'extra_point', 'count': 438}"
6,"{'play_type': 'pass', 'count': 7420}"
7,"{'play_type': 'run', 'count': 4510}"
8,"{'play_type': 'kickoff', 'count': 968}"


In [6]:
mex = MetricsExecutor(df_twofive)

print(
    mex.df
      .select(pl.col("play_type").value_counts().alias("vc"))
      .unnest("vc")
      .sort("play_type")
)
# Expect play_type values like: "pass", "rush", (maybe None), and special teams if you didn't drop them

df_rush = mex._filter_kind(mex.df, "rush")
print("rush rows:", df_rush.height) 

shape: (3, 2)
┌───────────┬───────┐
│ play_type ┆ count │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ null      ┆ 4263  │
│ pass      ┆ 7420  │
│ rush      ┆ 4510  │
└───────────┴───────┘
rush rows: 4510


In [7]:
# --- Compare counts via Pandas vs Polars to ensure consistency ---
import polars as pl

# --- Polars value counts ---
vc_pl = (
    df_twofive
    .select(pl.col("season_type").value_counts().alias("vc"))  # returns a struct
    .unnest("vc")                                              # flatten struct -> two columns
)

# Handle Polars version differences ("count" vs "counts")
count_col = "count" if "count" in vc_pl.columns else "counts"

polars_vc = (
    vc_pl.sort("season_type")
         .to_pandas()
         .set_index("season_type")[count_col]
)

# --- Pandas value counts ---
pdf = df_twofive.to_pandas()
pandas_vc = pdf["season_type"].value_counts(dropna=False).sort_index()

print("Value counts (Polars):")
print(polars_vc.sort_index())

print("\nValue counts (Pandas):")
print(pandas_vc)

# --- Null counts for key columns ---
cols = ["epa", "pass", "rush"]

# Polars null counts
polars_null_counts = (
    df_twofive
    .select([pl.col(c).is_null().sum().alias(c) for c in cols])
    .to_pandas()
    .iloc[0]
)

# Pandas null counts
pandas_null_counts = pdf[cols].isna().sum()

print("\nNull counts (Polars):")
print(polars_null_counts)

print("\nNull counts (Pandas):")
print(pandas_null_counts)

Value counts (Polars):
season_type
REG    16193
Name: count, dtype: uint32

Value counts (Pandas):
REG    16193
Name: season_type, dtype: int64

Null counts (Polars):
epa     188
pass      0
rush      0
Name: 0, dtype: uint32

Null counts (Pandas):
epa     188
pass      0
rush      0
dtype: int64


## Build Executor

In [8]:
# B. Count of rush-flag vs play_type==rush
diag = {
    "rows_total": df_twofive.height,
    "rush_flag_rows": df_twofive.filter(pl.col("rush") == 1).height if "rush" in df_twofive.columns else "no 'rush' col",
    "play_type_rush_rows": df_twofive.filter(pl.col("play_type").str.to_lowercase().str.strip_chars() == "rush").height
}
diag

{'rows_total': 16193, 'rush_flag_rows': 4510, 'play_type_rush_rows': 0}

In [9]:
ex = MetricsExecutor(df_twofive)
print('Executor ready.')

Executor ready.


## Tests Follow 

#### EPA Per Play, Ranked

In [10]:
df_out, summary = ex.execute('ranked epa per play')
display(df_out.to_pandas())
print(summary)

,rank,team,epa_per_play,plays
0,1,ind,0.168236,475
1,2,dal,0.154035,523
2,3,kc,0.121330,484
3,4,gb,0.120480,410
4,5,det,0.119429,476
5,6,buf,0.117892,502
6,7,was,0.097276,474
7,8,pit,0.092975,440
8,9,sea,0.078156,453
9,10,ne,0.067629,477


Computed team-level offensive overall EPA/play. Rank 1 is highest (best) for offense. Includes a 'plays' column for sample size context.


#### Defensive EPA Per Play

In [11]:
df_out, summary = ex.execute('ranked defensive epa per play allowed')
display(df_out.to_pandas())
print(summary)

,rank,team,epa_per_play,plays
0,1,hou,-0.147726,388
1,2,min,-0.095579,408
2,3,den,-0.064237,493
3,4,la,-0.060429,521
4,5,sea,-0.041448,519
5,6,jax,-0.030929,472
6,7,ind,-0.023216,482
7,8,atl,-0.020478,358
8,9,det,-0.011154,463
9,10,lac,-0.001603,474


Computed team-level defensive overall EPA/play. Rank 1 is lowest (best) for defense. Includes a 'plays' column for sample size context.


#### Rushing EPA per play

In [12]:
df_out, summary = ex.execute('ranked rushing epa per play')
display(df_out.to_pandas())
print(summary)

,rank,team,epa_per_play,plays
0,1,ind,0.089640,167
1,2,car,0.039933,159
2,3,kc,0.019889,120
3,4,dal,0.003088,140
4,5,buf,0.000591,159
5,6,phi,-0.001926,139
6,7,gb,-0.003847,140
7,8,mia,-0.008075,111
8,9,pit,-0.009222,134
9,10,cle,-0.020486,141


Computed team-level offensive rush EPA/play. Rank 1 is highest (best) for offense. Includes a 'plays' column for sample size context.


#### Offensive Success Rate

In [13]:
df_out, summary = ex.execute('offensive success rate')
display(df_out.to_pandas())
print(summary)

,rank,team,success_rate,plays
0,1,ind,0.538136,475
1,2,det,0.522199,476
2,3,la,0.513859,474
3,4,was,0.510593,474
4,5,kc,0.506250,484
5,6,gb,0.501229,410
6,7,sf,0.500000,529
7,8,sea,0.495556,453
8,9,pit,0.495392,440
9,10,buf,0.493976,502


Computed team-level offensive overall success rate (share of plays marked successful; fallback is EPA>0). Rank 1 is highest for offense.


#### Defensive Success Rate

In [14]:
df_out, summary = ex.execute('defensive success rate')
display(df_out.to_pandas())
print(summary)

,rank,team,success_rate,plays
0,1,den,0.414286,493
1,2,hou,0.420779,388
2,3,car,0.433107,444
3,4,min,0.433168,408
4,5,cle,0.434783,463
5,6,la,0.438462,521
6,7,sea,0.441860,519
7,8,nyj,0.444672,493
8,9,lac,0.448203,474
9,10,was,0.452525,499


Computed team-level defensive overall success rate (share of plays marked successful; fallback is EPA>0). Rank 1 is lowest for defense.


## EPA Per Dropback

In [22]:

mex = MetricsExecutor(df_twofive)

# Run the query
out_df, summary = mex.execute("ranked offensive EPA per dropback")

# Print both to verify
print(out_df)
print(summary)

# Basic validation
assert isinstance(out_df, pl.DataFrame)
assert "epa_per_dropback" in out_df.columns
assert "dropbacks" in out_df.columns
assert out_df.height > 0, "No rows returned — filtering may have failed."

print("✅ EPA per dropback test passed.")

out_pd = out_df.to_pandas()
display(out_pd)

shape: (32, 4)
┌──────┬──────┬──────────────────┬───────────┐
│ rank ┆ team ┆ epa_per_dropback ┆ dropbacks │
│ ---  ┆ ---  ┆ ---              ┆ ---       │
│ u32  ┆ str  ┆ f32              ┆ u32       │
╞══════╪══════╪══════════════════╪═══════════╡
│ 1    ┆ sea  ┆ 0.317287         ┆ 173       │
│ 2    ┆ det  ┆ 0.316514         ┆ 187       │
│ 3    ┆ ne   ┆ 0.310361         ┆ 219       │
│ 4    ┆ gb   ┆ 0.309409         ┆ 166       │
│ 5    ┆ tb   ┆ 0.293293         ┆ 223       │
│ …    ┆ …    ┆ …                ┆ …         │
│ 28   ┆ lv   ┆ -0.056172        ┆ 209       │
│ 29   ┆ nyg  ┆ -0.068673        ┆ 236       │
│ 30   ┆ cin  ┆ -0.115053        ┆ 275       │
│ 31   ┆ ten  ┆ -0.254228        ┆ 234       │
│ 32   ┆ cle  ┆ -0.265621        ┆ 267       │
└──────┴──────┴──────────────────┴───────────┘
Computed team-level offensive EPA per dropback. Rank 1 is highest (best) for offense. Includes a 'dropbacks' column for sample size context.
✅ EPA per dropback test passed.


,rank,team,epa_per_dropback,dropbacks
0,1,sea,0.317287,173
1,2,det,0.316514,187
2,3,ne,0.310361,219
3,4,gb,0.309409,166
4,5,tb,0.293293,223
5,6,dal,0.248971,247
6,7,ind,0.245834,201
7,8,buf,0.219069,218
8,9,kc,0.213900,250
9,10,was,0.191740,206


## Tiny Synthetic Unit Test (Deterministic)

In [ ]:
toy = pl.DataFrame({
    'posteam': ['A','A','B','B','C','C'],
    'defteam': ['X','Y','X','Y','X','Y'],
    'epa':     [0.2, -0.1, 0.05, 0.30, -0.20, 0.10],
    'play_type': ['pass','rush','pass','rush','pass','rush'],
    'pass':   [1,0,1,0,1,0],
    'rush':   [0,1,0,1,0,1],
})
ex_toy = MetricsExecutor(toy)

out, _ = ex_toy.execute('ranked epa per play')
vals = dict(zip(out['team'].to_list(), [round(x,3) for x in out['epa_per_play'].to_list()]))
assert vals == {'B': 0.175, 'A': 0.05, 'C': -0.05}, f"unexpected offense epa: {vals}"

out_d, _ = ex_toy.execute('ranked defensive epa per play allowed')
vals_d = dict(zip(out_d['team'].to_list(), [round(x,3) for x in out_d['epa_per_play'].to_list()]))
assert vals_d == {'X': 0.017, 'Y': 0.1}, f"unexpected defense epa: {vals_d}"

print('✓ Synthetic tests passed.')